# Notebook 2: ML Strategy (Logistic Regression + LightGBM)

**Purpose**: Walk-forward training and inference using LightGBM (main) and Logistic Regression (baseline). Generates BUY/SELL/HOLD signals, backtests, and saves results.

**Prerequisites**: Run `0_data_and_features.ipynb` first.

**GPU needed**: No (CPU only, ~20-25 min)

In [10]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from config import (
    ALL_TICKERS, FEATURES_DIR, CAPITAL_PER_ASSET, FEATURE_COLS,
    ML_TRAIN_WINDOW, ML_RETRAIN_FREQ, SAMPLE_WEIGHT_HALFLIFE,
    OPTUNA_TRIALS,
)
from utils import (
    setup_logging, create_directories, load_exchange_rates,
    simulate_trading, compute_metrics, save_results, is_indian_stock,
)

logger = setup_logging('ml_strategy')
create_directories()

# load common start date
with open(os.path.join('data', 'common_start_date.json'), 'r') as f:
    common_start = json.load(f)['common_start_date']
print(f'Common backtest start: {common_start}')

exchange_rates = load_exchange_rates()
print('Setup complete.')

Common backtest start: 2022-04-30
Setup complete.


## 1. Load Feature Data

In [11]:
def load_features(ticker):
    """Load pre-computed features for a single ticker."""
    safe_name = ticker.replace('=', '_').replace('-', '_')
    path = os.path.join(FEATURES_DIR, f'{safe_name}_features.parquet')
    return pd.read_parquet(path)

feature_data = {}
for ticker in ALL_TICKERS:
    feature_data[ticker] = load_features(ticker)
print(f'Loaded features for {len(feature_data)} assets.')

Loaded features for 20 assets.


## 2. Sample Weighting (Exponential Decay)

In [12]:
def compute_sample_weights(dates, half_life=SAMPLE_WEIGHT_HALFLIFE):
    """
    Compute exponential decay weights so recent data has higher importance.
    half_life: number of days after which weight drops to 50%.
    """
    max_date = dates.max()
    days_ago = (max_date - dates).days.values   # ← fixed: no .dt
    weights = np.exp(-0.693 * days_ago / half_life)
    return weights

## 3. LightGBM Hyperparameter Tuning with Optuna

In [13]:
def tune_lightgbm(X_train, y_train, weights, n_trials=OPTUNA_TRIALS):
    """
    Use Optuna to find best LightGBM hyperparameters.
    Uses last 20% of training data as validation (time-ordered, no shuffle).
    """
    # time-ordered validation split
    split_idx = int(len(X_train) * 0.8)
    X_tr, X_val = X_train[:split_idx], X_train[split_idx:]
    y_tr, y_val = y_train[:split_idx], y_train[split_idx:]
    w_tr, w_val = weights[:split_idx], weights[split_idx:]

    def objective(trial):
        params = {
            'objective': 'multiclass',
            'num_class': 3,
            'metric': 'multi_logloss',
            'verbosity': -1,
            'learning_rate': 0.05,
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr, sample_weight=w_tr,
            eval_set=[(X_val, y_val)],
            eval_sample_weight=[w_val],
            callbacks=[lgb.early_stopping(20, verbose=False)],
        )
        preds = model.predict(X_val)
        accuracy = (preds == y_val).mean()
        return accuracy

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

## 4. Walk-Forward ML Training & Inference

In [14]:
def walk_forward_ml(df, model_type='lightgbm', train_window=ML_TRAIN_WINDOW,
                    retrain_freq=ML_RETRAIN_FREQ):
    """
    Walk-forward training and prediction for ML models.
    
    Process:
    1. Start after train_window days of data
    2. Train model on past train_window days
    3. Predict for next retrain_freq days
    4. Slide window forward, retrain, repeat
    
    Returns: Series of 'BUY'/'SELL'/'HOLD' signals with DatetimeIndex.
    """
    # get valid rows (no NaN in features or target)
    valid_mask = df[FEATURE_COLS].notna().all(axis=1) & df['target'].notna()
    valid_df = df[valid_mask].copy()
    
    if len(valid_df) < train_window + retrain_freq:
        logger.warning(f'Not enough data for walk-forward. Returning HOLD.')
        return pd.Series('HOLD', index=df.index)
    
    dates = valid_df.index
    X_all = valid_df[FEATURE_COLS].values
    y_all = valid_df['target'].values.astype(int)
    
    all_signals = pd.Series('HOLD', index=df.index)  # default to HOLD
    signal_map = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}
    
    # walk forward
    start_idx = train_window
    last_trained_idx = -1
    model = None
    scaler = None
    
    while start_idx < len(valid_df):
        # determine train/test boundaries
        train_end = start_idx
        train_start = max(0, train_end - train_window)
        test_end = min(len(valid_df), start_idx + retrain_freq)
        
        # retrain if needed
        if start_idx - last_trained_idx >= retrain_freq or model is None:
            X_train = X_all[train_start:train_end]
            y_train = y_all[train_start:train_end]
            train_dates = dates[train_start:train_end]
            
            # normalize features
            scaler = RobustScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            
            # sample weights (exponential decay)
            weights = compute_sample_weights(pd.DatetimeIndex(train_dates))
            
            try:
                if model_type == 'lightgbm':
                    # tune hyperparams
                    best_params = tune_lightgbm(X_train_scaled, y_train, weights)
                    params = {
                        'objective': 'multiclass',
                        'num_class': 3,
                        'metric': 'multi_logloss',
                        'verbosity': -1,
                        'learning_rate': 0.05,
                        **best_params,
                    }
                    model = lgb.LGBMClassifier(**params)
                    model.fit(X_train_scaled, y_train, sample_weight=weights)
                else:
                    # logistic regression baseline
                    model = LogisticRegression(
                        max_iter=1000, multi_class='multinomial',
                        solver='lbfgs', C=1.0,
                    )
                    model.fit(X_train_scaled, y_train, sample_weight=weights)
                    
                last_trained_idx = start_idx
            except Exception as e:
                logger.error(f'Training failed at idx {start_idx}: {e}')
                start_idx += retrain_freq
                continue
        
        # predict on test window
        X_test = X_all[start_idx:test_end]
        X_test_scaled = scaler.transform(X_test)
        preds = model.predict(X_test_scaled)
        
        # map predictions to signal strings
        test_dates = dates[start_idx:test_end]
        for date, pred in zip(test_dates, preds):
            all_signals.loc[date] = signal_map.get(int(pred), 'HOLD')
        
        start_idx = test_end
    
    # shift signals by 1 day to prevent look-ahead
    all_signals = all_signals.shift(1).fillna('HOLD')
    
    return all_signals

## 5. Run LightGBM Strategy

In [15]:
lgbm_portfolios = {}
lgbm_orderbooks = {}
lgbm_metrics = {}

for i, (ticker, df) in enumerate(feature_data.items()):
    print(f'[{i+1}/{len(feature_data)}] LightGBM — {ticker}...', end=' ')
    
    try:
        signals = walk_forward_ml(df, model_type='lightgbm')
        
        exr = exchange_rates if is_indian_stock(ticker) else None
        ph, ob = simulate_trading(
            df, signals, ticker,
            initial_capital=CAPITAL_PER_ASSET,
            exchange_rates=exr,
            start_date=common_start,
        )
        
        lgbm_portfolios[ticker] = ph
        lgbm_orderbooks[ticker] = ob
        m = compute_metrics(ph)
        lgbm_metrics[ticker] = m
        
        print(f'Return: {m["total_return"]*100:+.1f}% | Sharpe: {m["sharpe_ratio"]:.2f} | '
              f'MaxDD: {m["max_drawdown"]*100:.1f}% | Trades: {m["num_trades"]}')
    except Exception as e:
        print(f'ERROR: {e}')
        logger.error(f'LightGBM failed for {ticker}: {e}')

print(f'\nLightGBM complete: {len(lgbm_metrics)} assets.')

[1/20] LightGBM — AAPL... Return: +1.0% | Sharpe: 0.07 | MaxDD: 18.1% | Trades: 692
[2/20] LightGBM — MSFT... Return: +9.5% | Sharpe: 0.32 | MaxDD: 15.9% | Trades: 697
[3/20] LightGBM — GOOGL... Return: -10.9% | Sharpe: -0.33 | MaxDD: 18.6% | Trades: 801
[4/20] LightGBM — AMZN... Return: +4.8% | Sharpe: 0.20 | MaxDD: 14.1% | Trades: 784
[5/20] LightGBM — NVDA... Return: +48.8% | Sharpe: 1.16 | MaxDD: 8.2% | Trades: 925
[6/20] LightGBM — META... Return: +39.8% | Sharpe: 0.98 | MaxDD: 11.0% | Trades: 856
[7/20] LightGBM — TSLA... Return: -4.1% | Sharpe: -0.14 | MaxDD: 15.3% | Trades: 919
[8/20] LightGBM — JPM... Return: +28.1% | Sharpe: 0.92 | MaxDD: 7.4% | Trades: 603
[9/20] LightGBM — JNJ... Return: -0.9% | Sharpe: 0.00 | MaxDD: 11.0% | Trades: 388
[10/20] LightGBM — V... Return: +13.4% | Sharpe: 0.51 | MaxDD: 7.6% | Trades: 421
[11/20] LightGBM — RELIANCE.NS... Return: +0.7% | Sharpe: 0.06 | MaxDD: 14.2% | Trades: 533
[12/20] LightGBM — TCS.NS... Return: -13.5% | Sharpe: -0.48 | MaxDD

## 6. Run Logistic Regression Baseline

In [16]:
logreg_portfolios = {}
logreg_orderbooks = {}
logreg_metrics = {}

for i, (ticker, df) in enumerate(feature_data.items()):
    print(f'[{i+1}/{len(feature_data)}] LogReg — {ticker}...', end=' ')
    
    try:
        signals = walk_forward_ml(df, model_type='logreg')
        
        exr = exchange_rates if is_indian_stock(ticker) else None
        ph, ob = simulate_trading(
            df, signals, ticker,
            initial_capital=CAPITAL_PER_ASSET,
            exchange_rates=exr,
            start_date=common_start,
        )
        
        logreg_portfolios[ticker] = ph
        logreg_orderbooks[ticker] = ob
        m = compute_metrics(ph)
        logreg_metrics[ticker] = m
        
        print(f'Return: {m["total_return"]*100:+.1f}% | Sharpe: {m["sharpe_ratio"]:.2f} | '
              f'MaxDD: {m["max_drawdown"]*100:.1f}% | Trades: {m["num_trades"]}')
    except Exception as e:
        print(f'ERROR: {e}')
        logger.error(f'LogReg failed for {ticker}: {e}')

print(f'\nLogistic Regression complete: {len(logreg_metrics)} assets.')

[1/20] LogReg — AAPL... Return: +20.9% | Sharpe: 0.56 | MaxDD: 12.2% | Trades: 731
[2/20] LogReg — MSFT... Return: +2.7% | Sharpe: 0.13 | MaxDD: 11.6% | Trades: 677
[3/20] LogReg — GOOGL... Return: +2.3% | Sharpe: 0.12 | MaxDD: 13.2% | Trades: 852
[4/20] LogReg — AMZN... Return: -3.2% | Sharpe: -0.09 | MaxDD: 17.6% | Trades: 730
[5/20] LogReg — NVDA... Return: +49.6% | Sharpe: 1.04 | MaxDD: 11.6% | Trades: 967
[6/20] LogReg — META... Return: +38.3% | Sharpe: 1.10 | MaxDD: 7.1% | Trades: 889
[7/20] LogReg — TSLA... Return: +3.5% | Sharpe: 0.16 | MaxDD: 12.6% | Trades: 972
[8/20] LogReg — JPM... Return: +16.6% | Sharpe: 0.57 | MaxDD: 12.6% | Trades: 512
[9/20] LogReg — JNJ... Return: +6.8% | Sharpe: 0.26 | MaxDD: 10.4% | Trades: 159
[10/20] LogReg — V... Return: +18.8% | Sharpe: 0.66 | MaxDD: 9.1% | Trades: 347
[11/20] LogReg — RELIANCE.NS... Return: -27.8% | Sharpe: -1.11 | MaxDD: 27.8% | Trades: 541
[12/20] LogReg — TCS.NS... Return: -9.6% | Sharpe: -0.31 | MaxDD: 12.7% | Trades: 240
[

## 7. Results Summary

In [17]:
# LightGBM vs LogReg comparison
comparison = []
for ticker in lgbm_metrics:
    lgbm_m = lgbm_metrics.get(ticker, {})
    lr_m = logreg_metrics.get(ticker, {})
    comparison.append({
        'Ticker': ticker,
        'LGBM Return': f"{lgbm_m.get('total_return', 0)*100:+.2f}%",
        'LGBM Sharpe': f"{lgbm_m.get('sharpe_ratio', 0):.2f}",
        'LGBM MaxDD': f"{lgbm_m.get('max_drawdown', 0)*100:.1f}%",
        'LogReg Return': f"{lr_m.get('total_return', 0)*100:+.2f}%",
        'LogReg Sharpe': f"{lr_m.get('sharpe_ratio', 0):.2f}",
    })

pd.DataFrame(comparison)

,Ticker,LGBM Return,LGBM Sharpe,LGBM MaxDD,LogReg Return,LogReg Sharpe
0,AAPL,+0.98%,0.07,18.1%,+20.91%,0.56
1,MSFT,+9.48%,0.32,15.9%,+2.72%,0.13
2,GOOGL,-10.87%,-0.33,18.6%,+2.29%,0.12
3,AMZN,+4.83%,0.20,14.1%,-3.23%,-0.09
4,NVDA,+48.80%,1.16,8.2%,+49.63%,1.04
5,META,+39.77%,0.98,11.0%,+38.29%,1.10
6,TSLA,-4.14%,-0.14,15.3%,+3.54%,0.16
7,JPM,+28.14%,0.92,7.4%,+16.55%,0.57
8,JNJ,-0.88%,0.00,11.0%,+6.77%,0.26
9,V,+13.40%,0.51,7.6%,+18.77%,0.66


## 8. Save Results

In [18]:
# save LightGBM results
save_results(lgbm_portfolios, lgbm_orderbooks, 'ml_lgbm')

# save LogReg results
save_results(logreg_portfolios, logreg_orderbooks, 'ml_logreg')

# save metrics
with open(os.path.join('results', 'ml_lgbm_metrics.json'), 'w') as f:
    json.dump(lgbm_metrics, f, indent=2)
with open(os.path.join('results', 'ml_logreg_metrics.json'), 'w') as f:
    json.dump(logreg_metrics, f, indent=2)

print('ML results saved to results/ directory.')
print('Notebook 2 complete.')

ML results saved to results/ directory.
Notebook 2 complete.
